# 🕵️ Agrupamento e Detecção de Anomalias com K-Means, DBSCAN, Agrupamento Espectral e One-Class SVM do Zero

Este notebook demonstra a aplicação prática de algoritmos não-supervisionados:
1. **Agrupamento e Detecção de Anomalias em Blobs Gaussianos**: Com `CustomKMeans`, `CustomDBSCAN` e `CustomOneClassSVM`.
2. **Agrupamento em Estruturas Não-Esféricas (Moons)**: Replicando o exemplo clássico de aula comparando K-Means, DBSCAN e **Agrupamento Espectral (CustomSpectralClustering)** baseado em grafos.

In [ ]:
import sys
sys.path.append('..')
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons
from src import (
    CustomKMeans,
    CustomDBSCAN,
    CustomSpectralClustering,
    CustomOneClassSVM,
    silhouette_values
)

## Parte 1: Agrupamento e Detecção de Anomalias em Blobs Gaussianos
Geramos dois agrupamentos gaussianos bem formados e adicionamos pontos anômalos espalhados.

In [ ]:
np.random.seed(42)
c1 = np.random.normal(loc=[2.0, 2.0], scale=0.6, size=(100, 2))
c2 = np.random.normal(loc=[7.0, 7.0], scale=0.8, size=(100, 2))
anomalies = np.random.uniform(low=-1.0, high=11.0, size=(10, 2))
X = np.vstack([c1, c2, anomalies])

plt.figure(figsize=(7, 7))
plt.scatter(X[:, 0], X[:, 1], color='gray', alpha=0.7, edgecolors='b')
plt.title('Conjunto de Dados Inicial (Blobs + Outliers)')
plt.grid(True)
plt.show()

### 1. K-Means nos Blobs

In [ ]:
kmeans = CustomKMeans(k=2, init_method='kmeans++', seed=42)
kmeans.fit(X)
labels_kmeans = kmeans.labels_
silhouette_scores = silhouette_values(X, labels_kmeans)
print(f'Inércia final do K-Means: {kmeans.inertia_:.4f}')
print(f'Silhueta média do agrupamento: {np.mean(silhouette_scores):.4f}')

### 2. DBSCAN nos Blobs

In [ ]:
dbscan = CustomDBSCAN(eps=1.2, min_samples=4)
dbscan.fit(X)
labels_db = dbscan.labels_
n_clusters = len(np.unique(labels_db[labels_db >= 0]))
n_noise = np.sum(labels_db == -1)
print(f'Clusters encontrados pelo DBSCAN: {n_clusters}')
print(f'Pontos classificados como ruído (-1): {n_noise}')

### 3. Detecção de Anomalias com One-Class SVM

In [ ]:
oc_svm = CustomOneClassSVM(nu=0.08, gamma=0.1)
oc_svm.fit(X)
preds_oc = oc_svm.predict(X)
anomalies_detected = np.sum(preds_oc == -1)
print(f'Anomalias detectadas pela One-Class SVM (-1): {anomalies_detected}')

### 4. Visualização Comparativa (Blobs)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(21, 7))

axes[0].scatter(X[:, 0], X[:, 1], c=labels_kmeans, cmap='viridis', s=40, edgecolors='k')
axes[0].scatter(kmeans.centroids_[:, 0], kmeans.centroids_[:, 1], color='red', marker='X', s=200, label='Centroides')
axes[0].set_title(f'K-Means (k=2) - Silhueta Média: {np.mean(silhouette_scores):.3f}')
axes[0].grid(True)
axes[0].legend()

unique_labels = np.unique(labels_db)
colors = [plt.cm.Spectral(each) for each in np.linspace(0, 1, len(unique_labels))]
for k_label, col in zip(unique_labels, colors):
    if k_label == -1:
        col = [0, 0, 0, 1]
    class_member_mask = (labels_db == k_label)
    axes[1].scatter(X[class_member_mask, 0], X[class_member_mask, 1], color=tuple(col), s=40, edgecolors='k')
axes[1].set_title(f'DBSCAN (eps=1.2, min_samples=4) - Clusters: {n_clusters}')
axes[1].grid(True)

normal_points = (preds_oc == 1)
outlier_points = (preds_oc == -1)
axes[2].scatter(X[normal_points, 0], X[normal_points, 1], color='tab:blue', label='Normal (Inlier)', s=40, edgecolors='k')
axes[2].scatter(X[outlier_points, 0], X[outlier_points, 1], color='tab:red', label='Outlier (Anomalia)', s=80, marker='o', edgecolors='k')
axes[2].set_title('Detecção de Anomalias com One-Class SVM (RBF)')
axes[2].grid(True)
axes[2].legend()

plt.tight_layout()
plt.show()

## Parte 2: Agrupamento em Estruturas Não-Esféricas (Exemplo de Aula)
Aqui replicamos o clássico problema de geometrias não-convexas onde o K-Means falha devido à distância euclidiana, e mostramos como o DBSCAN e o **Agrupamento Espectral** conseguem encontrar os grupos perfeitamente.

In [ ]:
# Geração do dataset de duas luas crescentes (moons) igual ao de aula
X_moons, y_moons = make_moons(n_samples=300, noise=0.05, random_state=42)

plt.figure(figsize=(7, 5))
plt.scatter(X_moons[:, 0], X_moons[:, 1], color='gray', edgecolors='k', alpha=0.8)
plt.title('Dataset Moons (Formato Não-Esférico)')
plt.grid(True)
plt.show()

### 1. Aplicação dos Modelos Customizados
Executamos os três algoritmos de agrupamento sobre o conjunto `X_moons`.

In [ ]:
# 1. K-Means com k=2
km_moons = CustomKMeans(k=2, init_method='kmeans++', seed=42)
km_moons.fit(X_moons)

# 2. DBSCAN com eps=0.2 e min_samples=7 (parâmetros de aula)
db_moons = CustomDBSCAN(eps=0.2, min_samples=7)
db_moons.fit(X_moons)

# 3. Agrupamento Espectral (Baseado em Grafos) com k=2 e similaridade gamma=10.0
spec_moons = CustomSpectralClustering(n_clusters=2, gamma=10.0, seed=42)
spec_moons.fit(X_moons)

print('Modelos ajustados com sucesso!')

### 2. Comparação Visual Lado a Lado (K-Means vs DBSCAN vs Espectral)
Visualizamos os agrupamentos resultantes. Nota-se como o K-Means corta os grupos linearmente devido à sua premissa geométrica, enquanto DBSCAN e Espectral separam as luas de maneira perfeita.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(21, 6))

# Plot K-Means
axes[0].scatter(X_moons[:, 0], X_moons[:, 1], c=km_moons.labels_, cmap='viridis', edgecolors='k', alpha=0.8)
axes[0].scatter(km_moons.centroids_[:, 0], km_moons.centroids_[:, 1], color='red', marker='X', s=200, label='Centroides')
axes[0].set_title('K-Means (Falha no formato)')
axes[0].grid(True)
axes[0].legend()

# Plot DBSCAN
db_labels = db_moons.labels_
axes[1].scatter(X_moons[db_labels >= 0, 0], X_moons[db_labels >= 0, 1], c=db_labels[db_labels >= 0], cmap='rainbow', edgecolors='k', alpha=0.8)
axes[1].scatter(X_moons[db_labels == -1, 0], X_moons[db_labels == -1, 1], color='black', label='Ruído', s=20, alpha=0.5)
axes[1].set_title('DBSCAN (eps=0.2, min_samples=7)')
axes[1].grid(True)
axes[1].legend()

# Plot Agrupamento Espectral
axes[2].scatter(X_moons[:, 0], X_moons[:, 1], c=spec_moons.labels_, cmap='coolwarm', edgecolors='k', alpha=0.8)
axes[2].set_title('Agrupamento Espectral (Sucesso via Grafo Laplacian)')
axes[2].grid(True)

plt.tight_layout()
plt.show()